In [1]:
import torch
import numpy as np
import random
import matplotlib.pyplot as plt
from pathlib import Path

from dmm.learning_algo_dir import LearningAlgorithm_dir
from dmm.learning_dir_detector import Learning_dir_detector
from dmm.learning_move_detector import Learning_move_detector
from dmm.learning_RTdetector import Learning_RTdetector
from dmm.utils import early_SSRT, RTgen_vs_SSD, DMM_vs_dim, plot_cs_ns, latent_traj_example, change_trial_ws, \
DMM_EV, PCA_vs_DMM, SSRT_plot, plot_cycle_shift_long, plot_cycle_shift_short, RT_pred_performance, consistent_sim,\
change_trial_cs, change_trial_cn, correlation_vs_gentime, cs_ws_SSD_hist, cn_ws_RT_hist
from dmm.dataset import process_data

In [2]:

# Piero:    2026-01-26-00h12_DKF_b4_Pz3_w3
# Cornelio: 2026-02-18-23h02_DKF_b4_3Cz3_w3
train_dir = "2026-01-26-00h12_DKF_b4_Pz3_w3"  
train_path = Path().cwd() / "saved_model" / train_dir

# set it to "" if you are testing Monkey C or if you are onyl testing Monkey P.
# Otherwise, set it to the folder name of Monkey C model, when reporting both monkeys' results.
with_Cornelio = "2026-02-18-23h02_DKF_b4_3Cz3_w3"

params = {
    "cfg": train_path / "config.ini",
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    "saved_dir": train_path
}

In [3]:
learning_algo = LearningAlgorithm_dir(params=params)
dmm = learning_algo.model

# Piero:    DKF_simple_final_epoch150
# Cornelio: DKF_simple_final_epoch178
weights_dir = "DKF_simple_final_epoch150.pt"

dmm.load_state_dict(torch.load(params['saved_dir'] / weights_dir, map_location=params['device']))
dmm.eval()
cfg = learning_algo.cfg

# Totale parametri
total_params = sum(p.numel() for p in dmm.parameters())
print(f"Totale parametri: {total_params}")

Totale parametri: 247252


In [4]:
# Load data
with np.load(params['saved_dir'] / "data_split.npz") as loaded_file:
    data = {key: loaded_file[key] for key in loaded_file.files}
    locals().update(data)  # Unpack all variables to local namespace

c_dim, z_dim, tau = cfg.getint('Network', 'c_dim'), cfg.getint('Network', 'z_dim'), cfg.getint('DataFrame', 'tau')

t = np.linspace(0, 255, 256, dtype=int)
t = t[::tau]

# Set random seeds
for seed_func in [random.seed, np.random.seed, torch.manual_seed]:
    seed_func(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Process test and train data
test_data = process_data(test_set, test_RT, test_SSD, test_direction, session_test, tau)
train_data = process_data(train_set, train_RT, train_SSD, train_direction, session_train, tau)

In [5]:
move_filename = "move_dict_z"
checkpoint_move = torch.load(params["saved_dir"] / move_filename, weights_only=False, map_location=params['device'])
move_learning_algo = Learning_move_detector(params=checkpoint_move["params"], device=params['device'], saved_dir=params["saved_dir"])
move_detector = move_learning_algo.load(move_filename)

Loaded model from epoch 69 with loss 0.0139 and accuracy 0.9767


In [6]:
RT_filename = "RT_dict_z"
checkpoint_RT = torch.load(params["saved_dir"] / RT_filename, weights_only=False, map_location=params['device'])
RT_learning_algo = Learning_RTdetector(params=checkpoint_RT["params"], device=params['device'], saved_dir=params["saved_dir"])
RT_detector = RT_learning_algo.load(RT_filename)

Loaded model from epoch 115 with loss 0.0002


In [7]:
dir_filename = "dir_dict_z"
checkpoint_dir = torch.load(params["saved_dir"] / dir_filename, weights_only=False, map_location=params['device'])
dir_learning_algo = Learning_dir_detector(params=checkpoint_dir["params"], device=params['device'], saved_dir=params["saved_dir"])
dir_detector = dir_learning_algo.load(dir_filename)

Loaded model from epoch 70 with loss 0.0451 and accuracy: 98.7864%


In [8]:
plt.style.use("paper.mplstyle")

comm_dict = {
    "train_dir": train_dir,
    "dmm": dmm,
    "saved_path": params["saved_dir"], 
    "move_detector": move_detector.to(params["device"]),
    "RT_detector": RT_detector.to(params["device"]),
    "dir_detector": dir_detector.to(params["device"]),
    "tau": tau,
    "z_dim": z_dim,
    "c_dim": c_dim,
    "device": params["device"],
    "with_Cornelio": with_Cornelio,
    "t": t,
    "z_lims": np.array([[-4.5, 2], [-4.5, 2.1], [-5, 2]]), #np.array([[-6, 8], [-9, 6], [-8, 8]]),
    "z_ticks": np.array([[-2, 0], [-2, 0], [-3, 0]]), #np.array([[-2, 1, 4], [-5, -1, 3], [-4, 0, 4]]),
}

### Evaluate the RT detector 

In [ ]:
diff_dict = {
    "data": test_data, 
    "n_trials": 1
}

RT_pred_performance(comm_dict, diff_dict)

### Display SSD and RT distribution

In [ ]:
diff_dict = {
    "data": train_data
}

cs_ws_SSD_hist(diff_dict)
cn_ws_RT_hist(diff_dict)

# START WITH THE PAPER FIGURES

### Fig. 1B. Funzionamento del modello

In [ ]:
# Fig.1B: Mostro una traiettoria esempio nello spazio latente
diff_dict = {
    "data": test_data,   
    "n_arrows": 16,
    "color_line": "green",
    "color_arrows": "#df9d07",
    "fontsize": 12,
    "lw_axis": 1.5,
    "lw_arrows": 1,
    "lw_line": 2,
    "offset": 1.1,
    "scale": 0.4,
    "origin": np.array([-1.5, -5.5, -3]),
    "elev": 70,
    "azim":-30,
}

latent_traj_example(comm_dict, diff_dict)

### Fig. 2 (A, B, C): DMM captures dynamics, PCA not 

In [ ]:
diff_dict = { 
    "x": np.array([0.3, 0.7]),
    "y_lim" : [(0, 1.1), (0, 1.1), (0.5, 1.1)],# (0.5, 1.1)],
    "y_ticks" : [(0, 1), (0, 1), (0.5, 1)],# (0.5, 1)],
    "inset_dim": [0.60, 0.65, 0.4, 0.35],
    "inset_font": 18,
    "width": 0.25,
    "color_PCA": "#ba0012",
    "color_DMM": "#df9d07"
}

PCA_vs_DMM(comm_dict, diff_dict)

### Fig. 3 (A, B, C): Identifying D=3 as the functional tipping point

In [ ]:
diff_dict = { 
    "n_trials": 10,
    "x": np.array([0.25, 0.5, 0.75]),
    "y_lim" : [(0, 1.1), (0, 1.1), (0.5, 1.05), (0.5, 1.05)],
    "y_ticks" : [(0, 1), (0, 1), (0.5, 1), (0.5, 1)],
    "width": 0.2,
    "metric": "MSE_pc",  # mean_NMSE, NMSE_mean, MSE_pc
    "inset_dim": [0.65, 0.7, 0.3, 0.28],
    "model_dirs": ["2026-01-17-18h40_DKF_b1_Pz2_w3", "2026-01-26-00h12_DKF_b4_Pz3_w3", "2026-01-23-10h22_DKF_b8_Pz4_w3"],
#     "model_dirs": ["2026-02-10-22h17_DKF_b1_Pz3_w3", "2026-01-26-00h12_DKF_b4_Pz3_w3", "2026-02-10-22h18_DKF_b10_Pz3_w3"],
#     "model_dirs": ["2026-02-15-21h35_DKF_b6_3Cz3_w3", "2026-02-16-07h47_DKF_b12_3Cz3_w3", "2026-02-17-09h40_DKF_b18_3Cz3_w3"],  # Cornelio_b12_3sessions
#     "model_dirs": ["2026-02-19-23h21_DKF_b1_3Cz3_w3", "2026-02-18-23h02_DKF_b4_3Cz3_w3", "2026-02-26-22h09_DKF_b10_3Cz3_w3"],  # 2026-02-18-23h09_DKF_b8_3Cz3_w3, 2026-02-18-23h03_DKF_b12_3Cz3_w3, 2026-02-18-23h07_DKF_b8_3Cz3_w3, 2026-02-19-23h22_DKF_b10_3Cz3_w3
    "colors": ["#ebc46a", "#df9d07", "#855e04"]#, ""]
}

DMM_vs_dim(comm_dict, diff_dict)

### Fig. 3D: Explained Variance

In [ ]:
diff_dict = {
    "model_dir": "2026-01-23-10h22_DKF_b8_Pz4_w3",
    "width": 0.15,
    "colors": ["#ebc46a", "#e5b038", "#df9d07", "#b27d05"],
    "color": "blue",
    "x_lims": (0.1, 0.9),
    "y_lims": (0, 0.6),
    "x_label" : ["$PC_1$", "$PC_2$", "$PC_3$", "$PC_4$"],
    "letter": "D",
    "alpha": 0.2,
    "delta_y": 0.03
}


DMM_EV(comm_dict, diff_dict)

### Fig. 4: Zero shot prediction of behaviour 

In [ ]:
diff_dict = {
    "n_trials": 1,
    "data": test_data,  # train_data, test_data
    "n_ticks": np.array([2, 2]),
    "sim_start": 56*5,
    "cmap": "viridis",
    "bins": 16,
    "alpha": 0.4,
    "x_ticks_hist": [500, 800],  # [400, 700]   [600, 800]
    "mean_z": True,
    "color_true": "#29B83B",
    "color_pred": "#df9d07",
    "eps": 1,
    "inset_dim": [0.65, 0.65, 0.35, 0.35],
    "inset_font": 13,
    "compute": False,
}

consistent_sim(comm_dict, diff_dict)

### Fig. 5: RT correlations between true and generated single trajectories

In [ ]:
diff_dict = {
    "n_trials": 20,
    "sim_start_array": np.arange(comm_dict["tau"], 201, comm_dict["tau"])*5,
    "data": test_data,
    "min_n": 120,
    "mean_corr": False,
    "figsize": (10, 6),
    "compute": False,
    "color_true": "#29B83B",
    "color_corr": "blue",
    "color_edge": "black",
    "num_bins": 16, 
    "alpha": 0.5,
    "h_hist": 0.009,
    "y_lim": 1
}

dict_corr = correlation_vs_gentime(comm_dict, diff_dict)

### Fig. 6 (A, B): State dependent stopping dynamics

In [ ]:
diff_dict = {
    "n_trials": 1,
    "SSD_early": 63,
    "SSD_late": 102,
    "cmap_ws": "Greens",  #YlOrBr
    "cmap_cs": "Reds",
    "color_stop_cs": "r",
    "color_stop_ws": "g",
    "color_line": 'black',
    "alpha_line": 0.5,
    "lw": 3,
    "lw_pc": 1,
    "xlims": [-1.85, 1.9],   # [-2.055, 1.9], [-3.2, 0], [0.5, 2.5]
    "min_grad": 0.4,
    "s": 10,
    "s_pc": 2, 
    "alpha": 0.4,
    "n_arrows": 15,
    "data": train_data,
    "axis": [0, 1],
    "axis_pca": [0, 2],
    "alpha_pc": 0.4,
    "inset_dim": [0.7, 0.06, 0.3, 0.3],
    "inset_font": 12,
    "xlims_pc": [-7, 7],
}

xx, yy = plot_cs_ns(comm_dict, diff_dict)

### Fig. 6 (C, D): Causal intervention

In [ ]:
diff_dict = {
    "n_trials": 1,
    "n_cont": 4,
    "RT": 98,   # 88,           
    "stop_array": np.array([180, 350]),
    "cmap": "Greens",
    "cmap_stop": "Reds",
    "color_stop": "r",
    "color_line": 'black',
    "alpha_line": 0.5,
    "lw": 4,
    "min_grad": 0.4,
    "min_grad_mod": 0.9,
    "lw": 3,
    "n_arrows_init": 15,
    "data": test_data,
    "axis": [0, 1],
    "line": [xx, yy],
}

change_trial_cn(comm_dict, diff_dict)

### Fig. 7: SSRT vs SSD matches human behaviour

In [ ]:
diff_dict = {
    "data": test_data,   # test_data, train_data
    "n_trials": 20,
    "chunk_size": 500, 
    "simuldistr": True,  
    "compute": False, 
    "SSD_short_list": [23, 28, 33, 38, 43, 48, 53, 58, 63, 68, 73, 78, 83, 88, 93, 98, 103, 108, 113, 118, 123, 128, 133, 138, 143],
    "color_Piero": "green",
    "ylims": [50, 300],
    "slice_start": 4,
    "slice_end": 16,
    "mean_z": False,
    "color_humans": "grey",
    "y_ticks": [100, 200, 300],
    "color_Cornelio": "red",
    "alpha": 0.3,
    "figsize": (10, 6)
}


early_SSRT(comm_dict, diff_dict)

### Fig. 8: delta RT vs SSD

In [ ]:
diff_dict = {
    "data": test_data,   # test_data, train_data
    "n_trials": 200,
    "chunk_size": 10, 
    "cmap": "viridis",
    "elinewidth": 1,
    "capsize": 9,
    "ms": 8,
    "min_ns": 10,
    "compute": False,
    "simulate_go": True,  
    "mean_z": False, 
    "SSD_list": [215, 315, 415, 515, 615],   # [43, 63, 83, 103, 123]
    "axis": [0, 2],
    "color_point": "#df9d07",
    "color_line": "black",
    "color_vline":"red",
    "size": 10,
    "xlims": [-300, 300],
    "ylims": [-50, 100],
    "ylims_inset": [-180, 0],
    "x_ticks": [200, 400, 600],
    "y_ticks": [-50, 0, 50],
    "y_ticks_inset": [-100, 0],
    "inset_dim": [0.55, 0.60, 0.45, 0.4],
    "inset_font": 15,
    "alpha": 0.5,
    "bins": 30,
    "figsize": (10, 6),
}

RTgen_vs_SSD(comm_dict, diff_dict)

### Fig. 9: SSRT vs mean RT

In [ ]:
diff_dict = {
    "data": train_data,   
    "n_trials": 20,
    "chunk_size": 1, 
    "elinewidth": 1,
    "capsize": 9,
    "ms": 8,
    "logit_move": True,  
    "RT_groups": 9,
    "move_frac": 0.5,
    "delta_max": 200,
    "compute": False, 
    "mean_z": False,
    "mean_z_flag": True,
    "cut_tail": True,
    "frac_tail": 0.05,
    "SSD_interval": [0, 60],
    "no_zero": True,
    "color_point": "#df9d07",
    "color_line": "black",
    "inset_dim": [0.62, 0.1, 0.38, 0.38],
    "inset_font": 15,
    "figsize": (10, 6)
}

SSRT_plot(comm_dict, diff_dict)

### Fig. 10A: Causal experiment. RT longening

In [ ]:
diff_dict = {
    "data": test_data,    
    "n_trials": 20,       # mostra un trial in sovraimpressione
    "stimulation_steps": 20,
    "l": 0.12,
    "step": 76,
    "bins": 20,
    "f": 0.1,
    "alpha_L1": 1e-3,
    "l1_ratio": 0.1,   # rapporto L1/L2
    "mean": True,
    "multi_direction":True, 
    "stimolate": True,
    "add_residual": True,
    "alpha": 0.4,
    "compute": False,
    "mean_trials":True,  # True mostra la z media di n trials, False mostra tutte le z degli n_trials
    "color_true": "#29B83B",
    "color_pred": "#df9d07",
    "x_ticks_hist": [400, 700],  # [400, 700]   [600, 800]
    "markersize": 5,
    "markeredgewidth": 2, 
    "color_edge": 'r',
    "cmap_stim": "bwr",
    "thr": -0.05,
    "inset_dim": [0.65, 0.7, 0.3, 0.3], 
}

plot_cycle_shift_long(comm_dict, diff_dict)

### Fig. 10B: Causal experiment. RT shortening

In [ ]:
diff_dict = {
    "data": test_data,    
    "n_trials": 20,
    "stimulation_steps": 20,
    "l": 0.16,
    "step": 76,
    "bins": 20,
    "f": 0.1,
    "alpha_L1": 1e-3,
    "l1_ratio": 0.1,   # rapporto L1/L2
    "mean": True,
    "multi_direction":True, 
    "stimolate": True,
    "add_residual": True,
    "alpha": 0.4,
    "compute": True,
    "mean_trials":True,  # True mostra la z media di n trials, False mostra tutte le z degli n_trials
    "color_true": "#29B83B",
    "color_pred": "#df9d07",
    "x_ticks_hist": [400, 700],  # P: [400, 700]  C: [600, 800]
    "markersize": 5,
    "markeredgewidth": 2, 
    "color_edge": 'r',
    "cmap_stim": "bwr",  # RdBu, bwr, seismic
    "thr": 0.05,
    "inset_dim": [0.05, 0.75, 0.25, 0.25], 
}


plot_cycle_shift_short(comm_dict, diff_dict)

### Supplementary

In [ ]:
diff_dict = {
    "n_trials": 1,
    "SSD": 62,
    "anticipation": 150,
    "cmap": "Greens",  #YlOrBr
    "cmap_go": "Reds",
    "color_fstop": "r",
    "color_stop": "g",
    "min_grad": 0.4,
    "lw": 3,
    "n_arrows_init": 15,
    "data": train_data,
    "axis": [0, 2],
}

change_trial_ws(comm_dict, diff_dict)

In [ ]:
diff_dict = {
    "n_trials": 1,
    "SSD": 103,  # 82
    "cmap": "Greens",
    "cmap_go": "Reds",
    "color_stop": "g",
    "min_grad": 0.4,
    "lw": 3,
    "n_arrows_init": 15,
    "data": test_data,
    "axis": [0, 2]
}

change_trial_cs(comm_dict, diff_dict)